# FIT5196 Assessment 1 — Group030 Solution
Complete structured parsing, transformation, reconciliation and validation evidence. **Members must add names and student IDs before submission.**

## 0. Configuration and reproducibility
All paths are relative/configurable; the workflow performs no network I/O.

In [1]:
from pathlib import Path
import json, hashlib, xml.etree.ElementTree as ET
import pandas as pd
GROUP_ID='Group030'; INPUT_DIR=Path('raw_input'); OUTPUT_DIR=Path('outputs'); TEMPLATE_DIR=Path('templates'); DICTIONARY_PATH=Path('public_data_dictionary.csv')
OUTPUT_DIR.mkdir(exist_ok=True)
from Group030_text_functions import *
print({'group':GROUP_ID,'input':str(INPUT_DIR),'output':str(OUTPUT_DIR),'pandas':pd.__version__})

{'group': 'Group030', 'input': 'raw_input', 'output': 'outputs', 'pandas': '3.0.5'}


### 0.2 Complete workflow implementation
This notebook is the canonical workflow. The following cell contains the complete structured parsing, normalisation, reconciliation, validation and export implementation. `Group030_solution.py` is exported from this cell; the notebook does not import that script.

In [2]:
"""Reproducible Group030 JSON/XML integration and validation workflow."""

from __future__ import annotations

import argparse
import json
import re
import xml.etree.ElementTree as ET
from pathlib import Path

import pandas as pd

from Group030_text_functions import (
    MISSING, build_latin_analysis, clean_narrative_text,
    contains_non_latin_script, extract_order_reference,
    extract_product_sku, extract_promo_code,
)

GROUP_ID = "Group030"
TABLES = ["orders", "order_items", "customers", "deliveries", "products", "product_reviews"]


def money(value):
    if isinstance(value, (int, float)): return float(value)
    return float(re.sub(r"[^0-9.+-]", "", str(value).replace(",", "")))


def boolean(value):
    if isinstance(value, bool): return value
    v = str(value).strip().lower()
    if v in {"true", "t", "yes", "y", "1"}: return True
    if v in {"false", "f", "no", "n", "0"}: return False
    raise ValueError(f"Unrecognised boolean: {value!r}")


def text(value):
    v = "" if value is None else str(value).strip()
    return v if v else MISSING


def date(value, dayfirst=False):
    return pd.to_datetime(value, dayfirst=dayfirst).strftime("%Y-%m-%d")


def timestamp(value, dayfirst=False):
    return pd.to_datetime(value, dayfirst=dayfirst).strftime("%Y-%m-%d %H:%M:%S")


def xml_record(element):
    return {child.tag: child.text for child in element}


def normalise_order(h, source):
    get = (lambda a, b: h[a]) if source == "json" else (lambda a, b: h.findtext(b))
    raw_note = get("customerNote", "Customer_Note")
    discount_raw = get("couponDiscount", "Coupon_Discount")
    discount = float(discount_raw) if source == "json" else money(discount_raw)
    return {
        "order_id": text(get("orderID", "Order_ID")),
        "source_system_record_id": text(get("sourceSystemRecordID", "Source_System_Record_ID")),
        "customer_id": text(get("customerID", "Customer_ID")),
        "order_timestamp": timestamp(get("orderTimestamp", "Order_Timestamp"), source == "xml"),
        "sales_channel": text(get("salesChannel", "Sales_Channel")),
        "payment_method": text(get("paymentMethod", "Payment_Method")),
        "currency": text(get("currency", "Currency")),
        "nearest_warehouse": text(get("nearestWarehouse", "Nearest_Warehouse")),
        "order_status": text(get("orderStatus", "Order_Status")),
        "delivery_charges": round(money(get("deliveryCharges", "Delivery_Charges")), 2),
        "coupon_code": text(get("couponCode", "Coupon_Code")),
        "coupon_discount": discount,
        "season": text(get("season", "Season")),
        "expedited_delivery": boolean(get("expeditedDelivery", "Expedited_Delivery")),
        "customer_lat": float(get("customerLat", "Customer_Lat")),
        "customer_long": float(get("customerLong", "Customer_Long")),
        "device_type": text(get("deviceType", "Device_Type")),
        "referral_source": text(get("referralSource", "Referral_Source")),
        "customer_note_clean": clean_narrative_text(raw_note),
        "promo_code": extract_promo_code(raw_note),
    }


def normalise_item(item, source):
    get = (lambda a, b: item[a]) if source == "json" else (lambda a, b: item.findtext(b))
    qty = int(get("quantity", "Quantity")); price = money(get("unitPrice", "Unit_Price"))
    return {"order_item_id": text(get("orderItemID", "Order_Item_ID")),
            "order_id": text(get("orderID", "Order_ID")), "product_id": text(get("productID", "Product_ID")),
            "quantity": qty, "unit_price": round(price, 2), "line_revenue": round(qty * price, 2)}


def normalise_delivery(d, source):
    get = (lambda a, b: d[a]) if source == "json" else (lambda a, b: d.findtext(b))
    return {"delivery_id": text(get("deliveryID", "Delivery_ID")), "order_id": text(get("orderID", "Order_ID")),
      "dispatch_date": date(get("dispatchDate", "Dispatch_Date"), source == "xml"),
      "promised_date": date(get("promisedDate", "Promised_Date"), source == "xml"),
      "delivered_date": date(get("deliveredDate", "Delivered_Date"), source == "xml"),
      "carrier": text(get("carrier", "Carrier")), "service_level": text(get("serviceLevel", "Service_Level")),
      "delivery_status": text(get("deliveryStatus", "Delivery_Status")), "delay_days": int(get("delayDays", "Delay_Days")),
      "on_time_in_full": boolean(get("onTimeInFull", "On_Time_In_Full")),
      "fulfilment_hours": int(get("fulfilmentHours", "Fulfilment_Hours")),
      "delivery_cost": round(money(get("deliveryCost", "Delivery_Cost")), 2),
      "delay_reason": text(get("delayReason", "Delay_Reason")), "promised_days": int(get("promisedDays", "Promised_Days")),
      "tracking_event_count": int(get("trackingEventCount", "Tracking_Event_Count")),
      "delivery_window": text(get("deliveryWindow", "Delivery_Window")),
      "shipping_distance_km": float(get("shippingDistanceKm", "Shipping_Distance_Km")),
      "signature_required": boolean(get("signatureRequired", "Signature_Required")),
      "estimated_carbon_kg": float(get("estimatedCarbonKg", "Estimated_Carbon_Kg")),
      "delivery_note_clean": clean_narrative_text(get("deliveryNoteClean", "Delivery_Note_Clean"))}


def normalise_review(r, source):
    get = (lambda a, b: r[a]) if source == "json" else (lambda a, b: r.findtext(b))
    raw = get("reviewText", "Review_Text"); clean = clean_narrative_text(raw)
    return {"review_id": text(get("reviewID", "Review_ID")), "order_id": text(get("orderID", "Order_ID")),
      "order_item_id": text(get("orderItemID", "Order_Item_ID")), "product_id": text(get("productID", "Product_ID")),
      "customer_id": text(get("customerID", "Customer_ID")),
      "review_timestamp": timestamp(get("reviewTimestamp", "Review_Timestamp"), source == "xml"),
      "language_code": text(get("languageCode", "Language_Code")), "rating": int(get("rating", "Rating")),
      "review_title": text(get("reviewTitle", "Review_Title")), "review_body_clean": clean,
      "review_body_latin_analysis": build_latin_analysis(clean),
      "verified_purchase": boolean(get("verifiedPurchase", "Verified_Purchase")),
      "helpful_votes": int(get("helpfulVotes", "Helpful_Votes")),
      "review_length_chars": 0 if clean == MISSING else len(clean),
      "review_word_count": 0 if clean == MISSING else len(clean.split()),
      "contains_non_latin_script": contains_non_latin_script(clean),
      "extracted_order_reference": extract_order_reference(raw), "extracted_product_sku": extract_product_sku(raw),
      "delivery_experience": text(get("deliveryExperience", "Delivery_Experience")),
      "value_experience": text(get("valueExperience", "Value_Experience")),
      "writing_style": text(get("writingStyle", "Writing_Style"))}


def reconcile(records, key, table, conflicts):
    canonical = {}
    for source, row in records:
        k = row[key]
        if k not in canonical: canonical[k] = (row, {source})
        else:
            old, sources = canonical[k]
            differences = {f: (old[f], row[f]) for f in row if old[f] != row[f]}
            if differences: conflicts.append({"table": table, "key": k, "differences": differences})
            sources.add(source)
    return [canonical[k][0] for k in sorted(canonical)]


def build_tables(input_dir, dictionary_path):
    with open(input_dir / f"{GROUP_ID}_commerce.json", encoding="utf-8") as f: js = json.load(f)
    root = ET.parse(input_dir / f"{GROUP_ID}_operations.xml").getroot()
    conflicts = []
    order_rows=[]; item_rows=[]; delivery_rows=[]; review_rows=[]
    for o in js["orders"]:
        order_rows.append(("JSON", normalise_order(o["header"], "json")))
        item_rows += [("JSON", normalise_item(x, "json")) for x in o["shoppingCart"]]
        if o.get("delivery"): delivery_rows.append(("JSON", normalise_delivery(o["delivery"], "json")))
    for o in root.findall("./Orders/Order"):
        order_rows.append(("XML", normalise_order(o.find("Header"), "xml")))
        item_rows += [("XML", normalise_item(x, "xml")) for x in o.findall("./Shopping_Cart/Item")]
        if o.find("Delivery") is not None: delivery_rows.append(("XML", normalise_delivery(o.find("Delivery"), "xml")))
    for r in js["productReviews"]: review_rows.append(("JSON", normalise_review(r, "json")))
    for r in root.findall("./ProductReviews/Review"): review_rows.append(("XML", normalise_review(r, "xml")))
    customers=[]
    cmap={"customerID":"customer_id","signupDate":"signup_date","loyaltyTier":"loyalty_tier","customerSegment":"customer_segment","ageBand":"age_band","preferredChannel":"preferred_channel","homeSuburb":"home_suburb","prior12MOrders":"prior_12m_orders","lifetimeValueBeforePeriod":"lifetime_value_before_period","marketingConsent":"marketing_consent","homePostcode":"home_postcode","homeState":"home_state","homeCountry":"home_country","preferredLanguage":"preferred_language","acquisitionSource":"acquisition_source","accountStatus":"account_status","preferredDevice":"preferred_device","emailDomain":"email_domain","householdSizeBand":"household_size_band","contactFrequencyPreference":"contact_frequency_preference"}
    for x in js["customerProfiles"]:
        row={v:x[k] for k,v in cmap.items()}; row["signup_date"]=date(row["signup_date"]); row["home_postcode"]=str(row["home_postcode"]); customers.append(row)
    products=[]
    pmap={"Product_ID":"product_id","Product_Name":"product_name","Category":"category","Brand":"brand","Unit_Price":"unit_price","Unit_Cost":"unit_cost","Launch_Year":"launch_year","Warranty_Months":"warranty_months","Weight_Kg":"weight_kg","Product_Sku":"product_sku","Subcategory":"subcategory","Model_Family":"model_family","Colour":"colour","Supplier_ID":"supplier_id","Supplier_Country":"supplier_country","Launch_Date":"launch_date","Tax_Category":"tax_category","Package_Type":"package_type","Recyclable_Packaging":"recyclable_packaging","Active_Flag":"active_flag","Product_Description":"product_description_clean"}
    for p in root.findall("./ProductCatalogue/Product"):
        x=xml_record(p); row={v:x.get(k) for k,v in pmap.items()}
        for f in ["unit_price","unit_cost","weight_kg"]: row[f]=money(row[f])
        for f in ["launch_year","warranty_months"]: row[f]=int(row[f])
        for f in ["recyclable_packaging","active_flag"]: row[f]=boolean(row[f])
        row["launch_date"]=date(row["launch_date"], True); row["product_description_clean"]=clean_narrative_text(row["product_description_clean"]); products.append(row)
    tables={"orders":pd.DataFrame(reconcile(order_rows,"order_id","orders",conflicts)),
      "order_items":pd.DataFrame(reconcile(item_rows,"order_item_id","order_items",conflicts)),
      "customers":pd.DataFrame(sorted(customers,key=lambda x:x["customer_id"])),
      "deliveries":pd.DataFrame(reconcile(delivery_rows,"delivery_id","deliveries",conflicts)),
      "products":pd.DataFrame(sorted(products,key=lambda x:x["product_id"])),
      "product_reviews":pd.DataFrame(reconcile(review_rows,"review_id","product_reviews",conflicts))}
    # Published arithmetic is derived from canonical item lines, not trusted source totals.
    sums=tables["order_items"].groupby("order_id",as_index=False).line_revenue.sum().rename(columns={"line_revenue":"order_price"})
    tables["orders"]=tables["orders"].merge(sums,on="order_id",validate="one_to_one")
    tables["orders"]["order_price"]=tables["orders"]["order_price"].round(2)
    tables["orders"]["tax_amount"]=(tables["orders"].order_price/11).round(2)
    tables["orders"]["order_total"]=(tables["orders"].order_price*(1-tables["orders"].coupon_discount/100)+tables["orders"].delivery_charges).round(2)
    dictionary=pd.read_csv(dictionary_path)
    for name,df in tables.items():
        cols=dictionary.loc[dictionary.output_table.eq(name)].sort_values("position").field_name.tolist()
        tables[name]=df[cols]
    source_key_sets={}
    for table, rows, key in [("orders",order_rows,"order_id"),("order_items",item_rows,"order_item_id"),("deliveries",delivery_rows,"delivery_id"),("product_reviews",review_rows,"review_id")]:
        source_key_sets[table]={s:[row[key] for src,row in rows if src==s] for s in ["JSON","XML"]}
    duplicates={table:{src:len(keys)-len(set(keys)) for src,keys in sources.items()} for table,sources in source_key_sets.items()}
    overlap={table:len(set(sources["JSON"]) & set(sources["XML"])) for table,sources in source_key_sets.items()}
    profile={"json":{"customers":len(js["customerProfiles"]),"orders":len(js["orders"]),
      "order_items":sum(len(x["shoppingCart"]) for x in js["orders"]),
      "deliveries":sum(bool(x.get("delivery")) for x in js["orders"]),"reviews":len(js["productReviews"])},
      "xml":{"orders":len(root.findall("./Orders/Order")),
      "order_items":len(root.findall("./Orders/Order/Shopping_Cart/Item")),
      "deliveries":len(root.findall("./Orders/Order/Delivery")),
      "products":len(root.findall("./ProductCatalogue/Product")),"reviews":len(root.findall("./ProductReviews/Review"))},
      "canonical":{k:len(v) for k,v in tables.items()}, "within_source_duplicates":duplicates,
      "cross_source_overlap":overlap, "conflicts":conflicts}
    return tables, profile


def validate(tables, dictionary, profile):
    rows=[]
    def add(cid, check, passed, observed, resolution="None required"):
        rows.append({"validation_id":cid,"check":check,"status":"PASS" if passed else "FAIL","observed_result":str(observed),"resolution_or_interpretation":resolution})
    expected=set(TABLES); add("VAL-SCHEMA-01","All six required tables are present",set(tables)==expected,sorted(tables))
    for name,df in tables.items():
        exp=dictionary[dictionary.output_table.eq(name)].sort_values("position").field_name.tolist()
        add(f"VAL-SCHEMA-{TABLES.index(name)+2:02d}",f"{name} columns match dictionary order",list(df)==exp,f"{name}: {len(df)} rows, {len(df.columns)} ordered columns")
        pk={"orders":"order_id","order_items":"order_item_id","customers":"customer_id","deliveries":"delivery_id","products":"product_id","product_reviews":"review_id"}[name]
        add(f"VAL-PK-{TABLES.index(name)+1:02d}",f"{name} primary key is complete and unique",df[pk].notna().all() and df[pk].is_unique,f"{name}.{pk}: missing={df[pk].isna().sum()}, duplicates={df[pk].duplicated().sum()}")
        required=dictionary[(dictionary.output_table.eq(name)) & (~dictionary.nullable.astype(bool))].field_name
        blank=sum((df[f].astype(str).str.strip()=="").sum() for f in required)
        add(f"VAL-MISS-{TABLES.index(name)+1:02d}",f"{name} required fields contain no empty strings",blank==0,f"{name}: empty required values={blank}")
        type_failures=[]
        for _,spec in dictionary[dictionary.output_table.eq(name)].iterrows():
            s=df[spec.field_name]; typ=spec.data_type
            ok=(pd.api.types.is_numeric_dtype(s) and not pd.api.types.is_bool_dtype(s)) if typ=="number" else pd.api.types.is_bool_dtype(s) if typ=="boolean" else s.astype(str).str.fullmatch(r"\d{4}-\d{2}-\d{2}").all() if typ=="date" else s.astype(str).str.fullmatch(r"\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}").all() if typ=="datetime" else True
            if not ok:type_failures.append(spec.field_name)
        add(f"VAL-TYPE-{TABLES.index(name)+1:02d}",f"{name} fields conform to dictionary data types",not type_failures,f"type failures={type_failures}")
    fks=[("orders","customer_id","customers","customer_id"),("order_items","order_id","orders","order_id"),("order_items","product_id","products","product_id"),("deliveries","order_id","orders","order_id"),("product_reviews","order_id","orders","order_id"),("product_reviews","order_item_id","order_items","order_item_id"),("product_reviews","product_id","products","product_id"),("product_reviews","customer_id","customers","customer_id")]
    for i,(ct,cf,pt,pf) in enumerate(fks,1):
        missing=set(tables[ct][cf])-set(tables[pt][pf]); add(f"VAL-FK-{i:02d}",f"{ct}.{cf} references {pt}.{pf}",not missing,f"orphan keys={len(missing)}")
    duplicate_counts=profile["within_source_duplicates"]
    add("VAL-DUP-01","Within-source duplicate primary keys are counted and reconciled",all(v>=0 for x in duplicate_counts.values() for v in x.values()) and not profile["conflicts"],duplicate_counts,"Identical normalised duplicates are collapsed by primary key; any differing field would appear in VAL-CONFLICT-01")
    add("VAL-OVERLAP-01","Cross-source key overlap is measured independently",all(v>0 for v in profile["cross_source_overlap"].values()),profile["cross_source_overlap"])
    add("VAL-CONFLICT-01","Overlapping normalised records contain no field conflicts",not profile["conflicts"],f"normalised cross-source conflicts={len(profile['conflicts'])}","Investigate every listed field conflict before submission")
    profile_names={"orders":"orders","order_items":"order_items","deliveries":"deliveries","product_reviews":"reviews"}
    for table,key in [("orders","order_id"),("order_items","order_item_id"),("deliveries","delivery_id"),("product_reviews","review_id")]:
        pkey=profile_names[table]; raw=profile["json"][pkey]+profile["xml"][pkey]; canonical=len(tables[table]); overlap=raw-canonical
        passed=0 <= overlap <= min(profile["json"][pkey],profile["xml"][pkey])
        add(f"VAL-FLOW-{TABLES.index(table)+2:02d}",f"{table} raw-to-canonical row equation holds",passed,f"JSON+XML={raw}, canonical={canonical}, overlap removed={overlap}, key={key}")
    items=tables["order_items"].groupby("order_id").line_revenue.sum().round(2)
    actual=tables["orders"].set_index("order_id").order_price
    add("VAL-ARITH-01","Order price equals sum of rounded line revenues",(actual-items).abs().le(.01).all(),f"max difference={(actual-items).abs().max():.4f}")
    o=tables["orders"]
    calc=(o.order_price*(1-o.coupon_discount/100)+o.delivery_charges).round(2)
    add("VAL-ARITH-02","Order total applies discount then delivery without adding GST",(o.order_total-calc).abs().le(.01).all(),f"max difference={(o.order_total-calc).abs().max():.4f}")
    add("VAL-ARITH-03","Tax is included GST equal to order_price/11",(o.tax_amount-o.order_price.div(11).round(2)).abs().le(.01).all(),"GST not added to total")
    numeric_ok=(o.order_price.ge(0)&o.delivery_charges.ge(0)&o.coupon_discount.between(0,100)&o.customer_lat.between(-90,90)&o.customer_long.between(-180,180)).all()
    add("VAL-RANGE-01","Order numeric values fall in sensible ranges",numeric_ok,"non-negative money; discount 0–100; valid coordinates")
    add("VAL-RANGE-02","Item quantity is positive and price non-negative",tables["order_items"].quantity.gt(0).all() and tables["order_items"].unit_price.ge(0).all(),f"min quantity={tables['order_items'].quantity.min()}, min price={tables['order_items'].unit_price.min()}")
    add("VAL-RANGE-03","Review rating is 1–5 and helpful votes non-negative",tables["product_reviews"].rating.between(1,5).all() and tables["product_reviews"].helpful_votes.ge(0).all(),f"rating={tables['product_reviews'].rating.min()}–{tables['product_reviews'].rating.max()}")
    categorical={"sales_channel":{"Web","Store","Mobile"},"currency":{"AUD"},"order_status":{"Completed"},"service_level":{"Express","Standard"},"delivery_status":{"Delivered"},"rating":{1,2,3,4,5}}
    cat_bad={f:sorted(set((tables["orders"] if f in tables["orders"] else tables["deliveries"] if f in tables["deliveries"] else tables["product_reviews"])[f])-allowed) for f,allowed in categorical.items()}
    add("VAL-CAT-01","Published structured categories use observed allowed vocabularies",all(not x for x in cat_bad.values()),cat_bad)
    d=tables["deliveries"].merge(o[["order_id","order_timestamp"]],on="order_id")
    # Dispatch is date-only, so compare calendar dates (same-day dispatch is valid).
    temporal=(pd.to_datetime(d.order_timestamp).dt.normalize()<=pd.to_datetime(d.dispatch_date)) & (pd.to_datetime(d.dispatch_date)<=pd.to_datetime(d.delivered_date))
    add("VAL-TIME-01","Order date <= dispatch <= delivered",temporal.all(),f"violations={(~temporal).sum()}")
    promised=(pd.to_datetime(d.dispatch_date)<=pd.to_datetime(d.promised_date)); delivered=pd.to_datetime(d.delivered_date); promised_date=pd.to_datetime(d.promised_date); expected_delay=(delivered-promised_date).dt.days.clip(lower=0)
    add("VAL-TIME-02","Promised date is not before dispatch",promised.all(),f"violations={(~promised).sum()}")
    delay_consistency=d.delay_days.eq(expected_delay) & d.on_time_in_full.eq(delivered.le(promised_date))
    add("VAL-TIME-03","Delay days and OTIF agree with promised/delivered dates",delay_consistency.all(),f"violations={(~delay_consistency).sum()}")
    rv=tables["product_reviews"].merge(o[["order_id","order_timestamp"]],on="order_id")
    rt=pd.to_datetime(rv.review_timestamp)>=pd.to_datetime(rv.order_timestamp)
    add("VAL-TIME-04","Review timestamp is not before order timestamp",rt.all(),f"violations={(~rt).sum()}")
    sentinel_fields=[("orders","coupon_code"),("orders","promo_code"),("product_reviews","extracted_order_reference"),("product_reviews","extracted_product_sku"),("product_reviews","review_body_latin_analysis")]
    empty=sum((tables[t][f].astype(str).str.strip()=="").sum() for t,f in sentinel_fields)
    add("VAL-TEXT-01","Prescribed missing strings use literal NaN, not empty",empty==0,f"empty prescribed strings={empty}")
    nonlatin=tables["product_reviews"].contains_non_latin_script
    add("VAL-TEXT-02","Multilingual reviews and non-Latin indicators are preserved",nonlatin.any(),f"non-Latin reviews={nonlatin.sum()} of {len(nonlatin)}")
    refs=tables["product_reviews"]
    order_pattern=r"^(?:NaN|[HC]ORD\d{6})$"; sku_pattern=r"^(?:NaN|SKU-[A-Z0-9]+)$"
    add("VAL-TEXT-03","Extracted order references follow bounded format",refs.extracted_order_reference.str.fullmatch(order_pattern).all(),f"invalid formats={(~refs.extracted_order_reference.str.fullmatch(order_pattern)).sum()}")
    add("VAL-TEXT-04","Extracted SKUs follow bounded format",refs.extracted_product_sku.str.fullmatch(sku_pattern).all(),f"invalid formats={(~refs.extracted_product_sku.str.fullmatch(sku_pattern)).sum()}")
    review_lengths=refs.review_body_clean.map(lambda x:0 if x==MISSING else len(x))
    add("VAL-TEXT-05","Review character counts derive from cleaned multilingual text",review_lengths.eq(refs.review_length_chars).all(),f"mismatches={(review_lengths!=refs.review_length_chars).sum()}")
    return pd.DataFrame(rows)


def main(input_dir=Path("raw_input"), output_dir=Path("outputs"), dictionary_path=Path("public_data_dictionary.csv")):
    output_dir.mkdir(parents=True,exist_ok=True)
    tables,profile=build_tables(Path(input_dir),Path(dictionary_path)); dictionary=pd.read_csv(dictionary_path)
    for name,df in tables.items(): df.to_csv(output_dir/f"{GROUP_ID}_{name}_standardised.csv",index=False,na_rep=MISSING)
    validations=validate(tables,dictionary,profile); validations.to_csv(output_dir/f"{GROUP_ID}_validation_register.csv",index=False)
    print(validations.to_string(index=False)); print("\nRow counts:", {k:len(v) for k,v in tables.items()})
    if (validations.status=="FAIL").any(): raise SystemExit("Validation failures require investigation")
    return tables, validations, profile


if __name__ == "__main__" and "get_ipython" not in globals():
    parser=argparse.ArgumentParser(); parser.add_argument("--input-dir",type=Path,default=Path("raw_input")); parser.add_argument("--output-dir",type=Path,default=Path("outputs")); parser.add_argument("--dictionary",type=Path,default=Path("public_data_dictionary.csv")); args=parser.parse_args()
    main(args.input_dir,args.output_dir,args.dictionary)


### 0.1 Package integrity
The archive, manifest and source filenames must all identify Group030; hashes verify that the allocated inputs were not altered.

In [3]:
manifest=json.load(open('A1_manifest.json'))
checks=[]
for entry in manifest['files']:
 p=Path('README.source.md') if entry['path']=='README.md' else Path(entry['path'])
 checks.append({'manifest_path':entry['path'],'local_path':str(p),'exists':p.exists(),'bytes':p.stat().st_size if p.exists() else None,'sha256_match':p.exists() and hashlib.sha256(p.read_bytes()).hexdigest()==entry['sha256']})
integrity=pd.DataFrame(checks);assert integrity.exists.all() and integrity.sha256_match.all();integrity

,manifest_path,local_path,exists,bytes,sha256_match
0,README.md,README.source.md,True,1059,True
1,public_data_dictionary.csv,public_data_dictionary.csv,True,11151,True
2,raw_input/Group030_commerce.json,raw_input/Group030_commerce.json,True,14284869,True
3,raw_input/Group030_operations.xml,raw_input/Group030_operations.xml,True,17863595,True


## 1. Parse and profile the two sources
Both documents are parsed with structured parsers. Regex is never used to reconstruct JSON/XML.

### 1.1 JSON structure and grain
`customerProfiles` is one customer per element; `orders` is one order with nested header, repeated cart items and one delivery; `productReviews` is one review. Candidate keys are customerID, orderID, orderItemID, deliveryID and reviewID. JSON uses ISO dates/timestamps, Python booleans, numeric AUD values and empty strings for optional text.

In [4]:
with open(INPUT_DIR/f'{GROUP_ID}_commerce.json',encoding='utf-8') as f: js=json.load(f)
json_profile=pd.DataFrame([{'collection':k,'python_type':type(v).__name__,'records':len(v) if isinstance(v,list) else 1,'top_fields':' | '.join(v[0].keys()) if isinstance(v,list) and v else ' | '.join(v.keys())} for k,v in js.items()])
json_profile

,collection,python_type,records,top_fields
0,customerProfiles,list,500,accountStatus | acquisitionSource | ageBand | ...
1,exportMetadata,dict,1,groupAlias | period | sourceSystem
2,orders,list,2818,delivery | header | shoppingCart
3,productReviews,list,3946,customerID | deliveryExperience | helpfulVotes...


### 1.2 XML structure and grain
`OperationsExport/Orders/Order` repeats orders; each contains Header, repeated Shopping_Cart/Item and one Delivery. ProductCatalogue/Product and ProductReviews/Review repeat at product and review grain. XML uses day-first dates, Y/N booleans, AUD labels, comma thousands separators, percentage strings and empty elements.

In [5]:
root=ET.parse(INPUT_DIR/f'{GROUP_ID}_operations.xml').getroot()
xml_profile=pd.DataFrame([('Orders/Order','order',len(root.findall('./Orders/Order'))),('Shopping_Cart/Item','order item',len(root.findall('./Orders/Order/Shopping_Cart/Item'))),('Order/Delivery','delivery',len(root.findall('./Orders/Order/Delivery'))),('ProductCatalogue/Product','product',len(root.findall('./ProductCatalogue/Product'))),('ProductReviews/Review','review',len(root.findall('./ProductReviews/Review')))],columns=['path','grain','records'])
xml_profile

,path,grain,records
0,Orders/Order,order,2818
1,Shopping_Cart/Item,order item,8885
2,Order/Delivery,delivery,2818
3,ProductCatalogue/Product,product,1000
4,ProductReviews/Review,review,3946


### 1.3 Source comparison, overlap and assumptions
JSON uniquely supplies customers; XML uniquely supplies products. Orders, items, deliveries and reviews overlap partially across sources. Comparable fields are normalised first, then compared by primary key. Equal records collapse; any differing non-missing value is a validation conflict. No source-precedence rule is used. IDs retain case/leading zeros; missing prescribed strings become literal `NaN`.

In [6]:
tables,profile=build_tables(INPUT_DIR,DICTIONARY_PATH)
pd.DataFrame(profile['json'].items(),columns=['entity','JSON rows']).merge(pd.DataFrame(profile['xml'].items(),columns=['entity','XML rows']),on='entity',how='outer').fillna(0), profile['canonical']

(        entity  JSON rows  XML rows
 0    customers      500.0       0.0
 1   deliveries     2818.0    2818.0
 2  order_items     8823.0    8885.0
 3       orders     2818.0    2818.0
 4     products        0.0    1000.0
 5      reviews     3946.0    3946.0,
 {'orders': 5000,
  'order_items': 15723,
  'customers': 500,
  'deliveries': 5000,
  'products': 1000,
  'product_reviews': 7000})

## 2. Source-to-target mapping
All 111 target fields retain template MAP IDs. Paths are structural, derivations specify formulas and all multi-source rows document comparison/conflict behaviour.

In [7]:
mapping=pd.read_csv('Group030_source_to_target_mapping.csv',keep_default_na=False)
core=['source_format','transformation_or_derivation','overlap_or_conflict_rule','notebook_evidence']
summary=mapping.groupby('output_table').agg(required_rows=('mapping_id','size'),core_complete=('mapping_id',lambda x:0))
summary['core_complete']=[mapping[mapping.output_table.eq(t)][core].ne('').all(axis=1).sum() for t in summary.index]
summary

,required_rows,core_complete
output_table,,
customers,20,20
deliveries,20,20
order_items,6,6
orders,23,23
product_reviews,21,21
products,21,21


## 3. Text and regex functions
Processing order is: structured field retrieval → reference extraction → entity decode/NFC → tags → published markers/URLs/emoji → complete reference wrapper → promo wrapper → whitespace/lowercase → literal `NaN`. `review_body_clean` preserves multilingual letters; Latin analysis is derived separately.

In [8]:
cases=pd.read_csv(TEMPLATE_DIR/'A1_public_text_test_cases.csv',keep_default_na=False)
results=[]
for _,x in cases.iterrows():
 actual=str(globals()[x.function](x.input_value));results.append({'case_id':x.case_id,'function':x.function,'actual':actual,'expected':x.expected_output,'status':'PASS' if actual==x.expected_output else 'FAIL'})
public_results=pd.DataFrame(results);public_results

,case_id,function,actual,expected,status
0,TXT-01,clean_narrative_text,leave at reception,leave at reception,PASS
1,TXT-02,extract_promo_code,B3SAVE-24,B3SAVE-24,PASS
2,TXT-03,clean_narrative_text,café setup was easy,café setup was easy,PASS
3,TXT-04,extract_order_reference,HORD123456,HORD123456,PASS
4,TXT-05,extract_product_sku,SKU-ABC123,SKU-ABC123,PASS
5,TXT-06,extract_order_reference,NaN,NaN,PASS
6,TXT-07,build_latin_analysis,service était bon,service était bon,PASS
7,TXT-08,contains_non_latin_script,True,True,PASS
8,TXT-09,build_latin_analysis,NaN,NaN,PASS
9,TXT-10,clean_narrative_text,reliable for daily use,reliable for daily use,PASS


In [9]:
student_tests=[('missing',clean_narrative_text(None),'NaN'),('Latin diacritic',build_latin_analysis('déjà 東京'),'déjà'),('non-Latin flag',contains_non_latin_script('déjà 東京'),True),('embedded order',extract_order_reference('XHORD123456'),'NaN'),('extended SKU',extract_product_sku('SKU-ABC_extra'),'NaN'),('extended promo',extract_promo_code('B3SAVE-24_more'),'NaN')]
pd.DataFrame(student_tests,columns=['case','actual','expected']).assign(status=lambda x:x.actual.eq(x.expected).map({True:'PASS',False:'FAIL'}))

,case,actual,expected,status
0,missing,NaN,NaN,PASS
1,Latin diacritic,déjà,déjà,PASS
2,non-Latin flag,True,True,PASS
3,embedded order,NaN,NaN,PASS
4,extended SKU,NaN,NaN,PASS
5,extended promo,NaN,NaN,PASS


## 4. Build the six standardised relational tables
The maintained implementation is `Group030_solution.py`; the following cells expose each table's grain, field order, row flow and material derivations.

### 4.1 Orders
One canonical order per `order_id`. Timestamps, percentages, booleans, coordinates and narrative are normalised. `order_price` is recomputed from rounded canonical lines; GST is included `order_price/11`; total applies discount then delivery.

In [10]:
tables['orders'].head(3), {'rows':len(tables['orders']),'unique_keys':tables['orders'].order_id.nunique(),'columns':tables['orders'].columns.tolist()}

(     order_id source_system_record_id customer_id      order_timestamp  \
 0  HORD000001        SRC-030-H-000001    CUS00225  2018-09-07 10:05:00   
 1  HORD000002        SRC-030-H-000002    CUS00099  2018-02-23 21:51:00   
 2  HORD000003        SRC-030-H-000003    CUS00238  2018-08-05 13:53:00   
 
   sales_channel payment_method currency nearest_warehouse order_status  \
 0         Store         PayPal      AUD          Thompson    Completed   
 1           Web         PayPal      AUD            Bakers    Completed   
 2           Web           Card      AUD         Nickolson    Completed   
 
    order_price  ...  tax_amount order_total  season  expedited_delivery  \
 0      6916.92  ...      628.81     6931.21  Spring               False   
 1      1867.88  ...      169.81     1878.88  Summer               False   
 2      7032.93  ...      639.36     7061.68  Winter                True   
 
    customer_lat customer_long  device_type  referral_source  \
 0    -37.882177    144.88

### 4.2 Order items
One row per `order_item_id`; nested arrays are flattened without joining reviews. `line_revenue = round(quantity × unit_price, 2)`.

In [11]:
tables['order_items'].head(3),tables['order_items'].agg({'quantity':['min','max'],'unit_price':['min','max'],'line_revenue':['min','max']})

(  order_item_id    order_id product_id  quantity  unit_price  line_revenue
 0   HITM0000001  HORD000001    PRD0054         2     2814.02       5628.04
 1   HITM0000002  HORD000001    PRD0466         1      520.19        520.19
 2   HITM0000003  HORD000001    PRD0628         1       62.19         62.19,
      quantity  unit_price  line_revenue
 min         1       15.23         15.23
 max         3     4182.34      12547.02)

### 4.3 Customers
One row per JSON customer. Postcodes remain strings so leading zeros would survive; consent is boolean and signup date is ISO.

In [12]:
tables['customers'].head(3),tables['customers'][['loyalty_tier','customer_segment','preferred_language']].describe()

(  customer_id signup_date loyalty_tier customer_segment age_band  \
 0    CUS00001  2016-03-17       Silver   Small Business    25-34   
 1    CUS00002  2017-12-10       Bronze       Mainstream    45-54   
 2    CUS00003  2014-09-05         Gold       Mainstream    35-44   
 
   preferred_channel home_suburb  prior_12m_orders  \
 0             Store    Hawthorn                 5   
 1             Store    Hawthorn                14   
 2            Mobile    St Kilda                 2   
 
    lifetime_value_before_period  marketing_consent home_postcode home_state  \
 0                       1876.75              False          3122        VIC   
 1                       5191.82               True          3122        VIC   
 2                       2135.34               True          3182        VIC   
 
   home_country preferred_language acquisition_source account_status  \
 0    Australia                 nl        Paid Search         Active   
 1    Australia                 de    

### 4.4 Deliveries
One completed delivery per `delivery_id`; XML/JSON dates and boolean alternatives are reconciled. Narrative is cleaned through the same bounded function.

In [13]:
tables['deliveries'].head(3),tables['deliveries'].groupby(['carrier','service_level']).size().rename('rows')

(  delivery_id    order_id dispatch_date promised_date delivered_date  \
 0  HDEL000001  HORD000001    2018-09-08    2018-09-13     2018-09-13   
 1  HDEL000002  HORD000002    2018-02-24    2018-03-01     2018-02-27   
 2  HDEL000003  HORD000003    2018-08-07    2018-08-12     2018-08-10   
 
           carrier service_level delivery_status  delay_days  on_time_in_full  \
 0         AusPost      Standard       Delivered           0             True   
 1  Direct Freight      Standard       Delivered           0             True   
 2         AusPost       Express       Delivered           0             True   
 
    fulfilment_hours  delivery_cost delay_reason  promised_days  \
 0                30          10.20         none              5   
 1                19           7.67         none              5   
 2                48          21.04         none              5   
 
    tracking_event_count delivery_window  shipping_distance_km  \
 0                     9       Afternoon    

### 4.5 Products
One XML catalogue row per product. AUD cost/price, dates, flags and product description are standardised.

In [14]:
tables['products'].head(3),tables['products'].groupby('category').agg(products=('product_id','size'),mean_price=('unit_price','mean'))

(  product_id      product_name    category   brand  unit_price  unit_cost  \
 0    PRD0001  Candle Bloom 100      Laptop  Candle     2393.92    1475.97   
 1    PRD0002     Vela Halo 101  Smartphone    Vela     1391.38     905.35   
 2    PRD0003  Candle Quest 102      Tablet  Candle      316.95     160.31   
 
    launch_year  warranty_months  weight_kg   product_sku  ... model_family  \
 0         2014               24      1.196  SKU-CAN00001  ...          Arc   
 1         2012               24      0.261  SKU-VEL00002  ...        Atlas   
 2         2017               24      0.610  SKU-CAN00003  ...        Bloom   
 
      colour supplier_id supplier_country launch_date  tax_category  \
 0    Silver      SUP001          Vietnam  2014-01-25  GST_STANDARD   
 1     Green      SUP002         Malaysia  2012-08-14  GST_STANDARD   
 2  Graphite      SUP003            Korea  2017-10-07  GST_STANDARD   
 
       package_type recyclable_packaging  active_flag  \
 0     Recycled box      

### 4.6 Product reviews
One canonical review per `review_id`; structured attributes are reconciled and raw text yields multilingual clean text, Latin analysis, references, flags and measures.

In [15]:
tables['product_reviews'].head(3),tables['product_reviews'].groupby(['language_code','contains_non_latin_script']).size().sort_values(ascending=False).head(12)

(    review_id    order_id order_item_id product_id customer_id  \
 0  HREV000001  HORD000001   HITM0000001    PRD0054    CUS00225   
 1  HREV000002  HORD000001   HITM0000003    PRD0628    CUS00225   
 2  HREV000003  HORD000001   HITM0000004    PRD0620    CUS00225   
 
       review_timestamp language_code  rating  \
 0  2018-09-15 11:01:00            en       5   
 1  2018-10-28 12:02:00            en       5   
 2  2018-10-13 13:03:00            en       3   
 
                           review_title  \
 0  a pleasant shared-screen experience   
 1     helpful summaries after exercise   
 2    protective case that travels well   
 
                                    review_body_clean  ... verified_purchase  \
 0  vela spark 153 has become the screen we use mo...  ...              True   
 1  i have used vela echo 727 after walks, short r...  ...              True   
 2  ⭐ vela atlas 719 has carried my tablet and cha...  ...              True   
 
    helpful_votes  review_length_cha

## 5. Reconciliation and relationships
Raw-to-canonical counts demonstrate partial overlap. Reconciliation is key-driven and deterministic; zero conflicts means overlapping values agreed after normalisation.

In [16]:
flow=[]
for table,rawname in [('orders','orders'),('order_items','order_items'),('deliveries','deliveries'),('product_reviews','reviews')]:
 raw=profile['json'][rawname]+profile['xml'][rawname];flow.append({'table':table,'JSON':profile['json'][rawname],'XML':profile['xml'][rawname],'raw_total':raw,'canonical':len(tables[table]),'overlap_removed':raw-len(tables[table])})
pd.DataFrame(flow),{'normalised_conflicts':len(profile['conflicts'])}

(             table  JSON   XML  raw_total  canonical  overlap_removed
 0           orders  2818  2818       5636       5000              636
 1      order_items  8823  8885      17708      15723             1985
 2       deliveries  2818  2818       5636       5000              636
 3  product_reviews  3946  3946       7892       7000              892,
 {'normalised_conflicts': 0})

## 6. Validation register
Each executable check reports a stable ID, observed result, PASS/FAIL and resolution. Coverage includes schema/order, missing representation, PK/FK, row flow, conflict detection, arithmetic, ranges, temporal order, references and multilingual behaviour.

In [17]:
dictionary=pd.read_csv(DICTIONARY_PATH);validation_register=validate(tables,dictionary,profile)
validation_register

,validation_id,check,status,observed_result,resolution_or_interpretation
0,VAL-SCHEMA-01,All six required tables are present,PASS,"['customers', 'deliveries', 'order_items', 'or...",None required
1,VAL-SCHEMA-02,orders columns match dictionary order,PASS,"orders: 5000 rows, 23 ordered columns",None required
2,VAL-PK-01,orders primary key is complete and unique,PASS,"orders.order_id: missing=0, duplicates=0",None required
3,VAL-MISS-01,orders required fields contain no empty strings,PASS,orders: empty required values=0,None required
4,VAL-TYPE-01,orders fields conform to dictionary data types,PASS,type failures=[],None required
5,VAL-SCHEMA-03,order_items columns match dictionary order,PASS,"order_items: 15723 rows, 6 ordered columns",None required
6,VAL-PK-02,order_items primary key is complete and unique,PASS,"order_items.order_item_id: missing=0, duplicat...",None required
7,VAL-MISS-02,order_items required fields contain no empty s...,PASS,order_items: empty required values=0,None required
8,VAL-TYPE-02,order_items fields conform to dictionary data ...,PASS,type failures=[],None required
9,VAL-SCHEMA-04,customers columns match dictionary order,PASS,"customers: 500 rows, 20 ordered columns",None required


In [18]:
validation_register.groupby('status').size(), validation_register[validation_register.status.ne('PASS')]

(status
 PASS    56
 dtype: int64,
 Empty DataFrame
 Columns: [validation_id, check, status, observed_result, resolution_or_interpretation]
 Index: [])

## 7. Deterministic export
Only dictionary fields, in published order, enter the six submitted CSVs. Reading with `keep_default_na=False` confirms literal `NaN` is retained.

In [19]:
for name,df in tables.items():df.to_csv(OUTPUT_DIR/f'{GROUP_ID}_{name}_standardised.csv',index=False,na_rep='NaN')
validation_register.to_csv(OUTPUT_DIR/f'{GROUP_ID}_validation_register.csv',index=False)
roundtrip={n:list(pd.read_csv(OUTPUT_DIR/f'{GROUP_ID}_{n}_standardised.csv',keep_default_na=False).columns)==list(tables[n].columns) for n in tables}
roundtrip

{'orders': True,
 'order_items': True,
 'customers': True,
 'deliveries': True,
 'products': True,
 'product_reviews': True}

## 8. Final reproducibility record
Fresh offline execution recreates six CSVs and all validation evidence without manual edits, certified counts, live services or absolute paths.

In [20]:
assert public_results.status.eq('PASS').all();assert validation_register.status.eq('PASS').all();assert all(roundtrip.values());print('FINAL STATUS: PASS — public tests, validation register and round-trip schemas')

FINAL STATUS: PASS — public tests, validation register and round-trip schemas
